# External Data Enrichment at Scale (Staff Engineer Guide)

## Context
You have a DataFrame with **1 Million+ rows** (claims). You need to enrich each row with a Hash ID fetched from an **external API** (`api.hashify.net`).

## The Problem
Calling an external API row-by-row in a distributed system is one of the most common performance killers. 

This notebook explores the evolution of a solution from **Naive** (Junior) to **Resilient & Scalable** (Staff/Principal).

### Approaches Covered
1.  **The Anti-Pattern**: Python UDF (Sequential Blocking).
2.  **The Intermediate Fix**: `mapPartitions` with Threading.
3.  **The Staff-Level Architecture**: 
    *   Async I/O
    *   Batching (The Contract Negotiation)
    *   Caching Strategies
4.  **The "Root Cause" Fix**: Local Computation.

In [ ]:
import time
import requests
import uuid
import pandas as pd
import hashlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField

# Initialize Spark
spark = SparkSession.builder \
    .appName("StaffLevel_API_Architecture") \
    .master("local[*]") \
    .getOrCreate()

# Generate 100 Sample Rows (We won't run 1M against the real API for this demo)
data = [(str(uuid.uuid4()),) for _ in range(20)]
df = spark.createDataFrame(data, ["claim_id"])
df.cache()
df.count()

## 1. The Anti-Pattern: Naive Python UDF

This is the initial implementation provided. It looks correct logically, but creates a massive bottleneck.

```mermaid
sequenceDiagram
    participant SparkWorker
    participant PythonUDF
    participant ExternalAPI

    Note over SparkWorker: Processing Partition 1
    loop For Every Row (1 Million Times)
        SparkWorker->>PythonUDF: Serialize Data
        PythonUDF->>ExternalAPI: HTTP GET (Waiting...)
        ExternalAPI-->>PythonUDF: Response (200ms)
        PythonUDF-->>SparkWorker: Return Result
    end
```

**Latency Calculation**:
*   1M rows * 200ms latency = 200,000 seconds = **55 Hours** (if single threaded).

In [ ]:
import logging
logger = logging.getLogger(__name__)

def get_hash_id(claim_id):
    """
    Fetches MD4 hash for a single claim_id from an external API.
    Returns the hash_id as a string, or empty string on error.
    """
    if not claim_id:
        return ""
    try:
        # NOTE: This is a GET request. 
        # In a real demo, we might mock this if the URL is unreachable, 
        # but here we use the code exactly as requested.
        url = f"https://api.hashify.net/hash/md4/hex?value={claim_id}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json().get("Digest", "")
        else:
            return ""
    except Exception as e:
        # logger.error(f"Error fetching hash for {claim_id}: {e}")
        return ""

# Wrap as UDF
udf_get_hash = F.udf(get_hash_id, StringType())

print("Running Naive UDF on 5 rows...")
start = time.time()
df.limit(5).withColumn("hash", udf_get_hash("claim_id")).show()
end = time.time()
print(f"Time taken for 5 rows: {end - start:.2f} seconds")
print(f"Projected time for 1M rows: {((end - start)/5 * 1_000_000) / 3600:.2f} Hours")

## 2. Intermediate Fix: `mapPartitions` & Connection Reuse

Instead of opening a new TCP connection for every row, we use `mapPartitions` to open one connection (Session) per partition and reuse it. 

**Improvement**: Removes TCP handshake overhead, but still blocked by request latency.

In [ ]:
def fetch_hashes_partition(iterator):
    # Initialize Session once per partition (worker task)
    session = requests.Session()
    results = []
    
    for row in iterator:
        claim_id = row.claim_id
        try:
            url = f"https://api.hashify.net/hash/md4/hex?value={claim_id}"
            resp = session.get(url)
            hash_val = resp.json().get("Digest", "")
        except:
            hash_val = ""
        
        results.append((claim_id, hash_val))
    
    return iter(results)

# Usage
# rdd = df.rdd.mapPartitions(fetch_hashes_partition)

## 3. Staff Level: Batching (The Architecture Fix)

The single most effective change is **negotiating a Batch API endpoint** with the backend team. Sending 1000 IDs in one request reduces network overhead by 99.9%.

```mermaid
graph LR
    subgraph "Naive Approach"
        A[Spark Worker] -- 1000 Req --> B(API)
    end
    
    subgraph "Staff Approach (Batching)"
        C[Spark Worker] -- 1 Req (Array of 1000) --> D(API)
        D -- 1 Resp (Map of ID->Hash) --> C
    end
```

In [ ]:
def fetch_batch_partition(iterator):
    batch_size = 100
    batch = []
    results = []
    session = requests.Session()
    
    for row in iterator:
        batch.append(row.claim_id)
        if len(batch) >= batch_size:
            # PSEUDOCODE: Assuming endpoint accepts POST with list
            # resp = session.post("https://api.../batch", json={"ids": batch})
            # results.extend(resp.json())
            batch = []
            
    # Flush remaining
    return iter(results)

## 4. Staff Level: Async I/O (Concurrency)

If Batching is impossible, we use Async I/O (Python `aiohttp` or Java `CompletableFuture` in Scala). This allows a single Spark core to fire off hundreds of requests without waiting for the response sequentially.

```mermaid
gantt
    title Sequential vs Async Execution
    dateFormat s
    axisFormat %S
    
    section Sequential
    Req 1 :a1, 0, 1s
    Req 2 :a2, 1, 1s
    Req 3 :a3, 2, 1s

    section Async
    Req 1 :b1, 0, 1s
    Req 2 :b2, 0.1, 1s
    Req 3 :b3, 0.2, 1s
```

## 5. Staff Level: Caching & Resilience

For expensive APIs, implement a **Look-aside Cache** (Redis/DynamoDB). 

1.  Check Redis for `claim_id`. 
2.  If Miss -> Call API -> Write to Redis.

This protects the API from traffic spikes during re-runs or backfills.

```mermaid
flowchart TD
    A[Spark Row] --> B{Check Redis}
    B -- Hit --> C[Return Hash]
    B -- Miss --> D[Call External API]
    D --> E[Write to Redis]
    E --> C
```

## 6. The "Root Cause" Fix (Best Practice)

**Question the Requirement**: Why are we calling an API to compute an MD4 hash?

MD4/MD5/SHA are standard algorithms. If the logic is deterministic and standard, **compute it locally**.

This eliminates the network dependency entirely.

In [ ]:
# Local Computation - 1000x Faster, Zero Latency, Zero API Cost
df.withColumn("hash_local", F.expr("hex(unhex(md5(claim_id)))")) # Spark doesn't have native MD4, but has MD5/SHA2.

# If MD4 is STRICTLY required, use a Python UDF or Scalar UDF (Java) to compute it locally.
# Do NOT go over the network for a math problem.
def local_md4(val):
    # Any local hashing library
    return hashlib.new('md4', str(val).encode('utf-16le')).hexdigest()

udf_md4 = F.udf(local_md4, StringType())
df.withColumn("hash_md4_local", udf_md4("claim_id")).show()